In [5]:
import os
import zipfile
import pandas as pd
import re
import gc  # Garbage collection to clear memory
import pickle
from tqdm import tqdm

# Function to extract CIK from the filename
def extract_cik(filename):
    match = re.search(r'_(\d+)_\d+-\d+', filename)
    if match:
        return match.group(1)
    return None

# Function to process zip files and extract 10-K information
def process_zip_files(folder_path, output_folder='processed_data_new', chunk_size=1000):
    # Create output folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)

    # Initialize an empty list to store the data for the dataframe
    data = []

    chunk_counter = 1  # Counter for chunk files

    # Use tqdm to show progress for the outer loop
    for zip_filename in tqdm(os.listdir(folder_path), desc="Processing ZIP files"):
        # Only process zip files
        if zip_filename.endswith('.zip'):
            zip_path = os.path.join(folder_path, zip_filename)

            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                # Extract all the contents of the zip file
                extract_folder = os.path.join(folder_path, zip_filename.replace('.zip', ''))
                os.makedirs(extract_folder, exist_ok=True)
                zip_ref.extractall(extract_folder)

                # Use tqdm to show progress for the inner loop
                for extracted_file in tqdm(zip_ref.namelist(), desc=f"Extracting files from {zip_filename}", leave=False):
                    # Only process .txt files that contain '10-K' in their name
                    if extracted_file.endswith('.txt') and '10-K' in extracted_file:
                        cik = extract_cik(extracted_file)  # Extract CIK from the filename
                        if cik:
                            # Read the contents of the .txt file
                            file_path = os.path.join(extract_folder, extracted_file)
                            with open(file_path, 'r', encoding='utf-8') as file:
                                note_text = file.read()

                            # Use re.search() to find the date (if it's anywhere in the filename)
                            date_match = re.search(r'(\d{8})(?=_10-K)', extracted_file)
                            date = date_match.group(1) if date_match else None

                            # If a date was found, convert it into YYYY-MM-DD format
                            if date:
                                date_obj = pd.to_datetime(date, format='%Y%m%d').strftime('%Y-%m-%d')
                            else:
                                date_obj = None

                            # Add the extracted data to the list
                            data.append([cik, date_obj, note_text])

                            # Save intermittently to avoid keeping everything in memory
                            if len(data) >= chunk_size:  # Save every chunk_size entries
                                # Save the data as a pickle file
                                temp_filename = os.path.join(output_folder, f"processed_data_chunk_{chunk_counter}.pkl")
                                with open(temp_filename, 'wb') as pkl_file:
                                    pickle.dump(data, pkl_file)

                                # Compress the pickle file into a .zip archive
                                with zipfile.ZipFile(temp_filename.replace('.pkl', '.zip'), 'w', zipfile.ZIP_DEFLATED) as zipf:
                                    zipf.write(temp_filename, os.path.basename(temp_filename))
                                    os.remove(temp_filename)  # Remove the uncompressed pickle file

                                # Reset data for the next chunk
                                data = []
                                chunk_counter += 1
                                gc.collect()  # Explicitly call garbage collection to free memory

    # Save any remaining data that was not saved
    if data:
        temp_filename = os.path.join(output_folder, f"processed_data_chunk_{chunk_counter}.pkl")
        with open(temp_filename, 'wb') as pkl_file:
            pickle.dump(data, pkl_file)

        # Compress the pickle file into a .zip archive
        with zipfile.ZipFile(temp_filename.replace('.pkl', '.zip'), 'w', zipfile.ZIP_DEFLATED) as zipf:
            zipf.write(temp_filename, os.path.basename(temp_filename))
            os.remove(temp_filename)  # Remove the uncompressed pickle file

        print(f"Remaining data saved to {temp_filename.replace('.pkl', '.zip')}")

    print(f"Processing completed. Data saved in chunks to {output_folder}")



In [6]:

# Example usage
folder_path = '../data/'  # Specify your folder path containing ZIP files
df = process_zip_files(folder_path)

Processing ZIP files:  20%|███████████▌                                              | 1/5 [41:15<2:45:00, 2475.11s/it]
racting files from 10-X_C_Zip-20250404T182603Z-001.zip:   0%|                                 | 0/1 [00:00<?, ?it/s]
Processing ZIP files:  40%|████████████████████████                                    | 2/5 [41:22<51:11, 1023.71s/it]
racting files from 10-X_C_Zip-20250404T182603Z-002.zip:   0%|                                 | 0/1 [00:00<?, ?it/s]
Processing ZIP files:  60%|████████████████████████████████████▌                        | 3/5 [41:31<18:40, 560.05s/it]
racting files from 10-X_C_Zip-20250404T182603Z-003.zip:   0%|                                 | 0/1 [00:00<?, ?it/s]
Processing ZIP files:  80%|████████████████████████████████████████████████▊            | 4/5 [41:38<05:42, 342.01s/it]
racting files from 10-X_C_Zip-20250404T182603Z-004.zip:   0%|                                 | 0/1 [00:00<?, ?it/s]
Processing ZIP files: 100%|█████████████████████████

Remaining data saved to processed_data_new\processed_data_chunk_41.zip
Processing completed. Data saved in chunks to processed_data_new


In [ ]:
df

In [ ]:
df.to_csv("full_10K_data.csv",index=False)